Merging the two datasets 

In [8]:
import pandas as pd
import numpy as np

# Load datasets
mental_df = pd.read_csv("Cleaned Mental Health Dataset.csv")
meal_df = pd.read_csv("Cleaned Meal Dataset.csv")

# Split meal dataset by category
healthy_meals = meal_df[meal_df['Health_Label'] == 'Healthy']
unhealthy_meals = meal_df[meal_df['Health_Label'] == 'Unhealthy']

assigned_meals = []

# Assign one meal to each person
for _, row in mental_df.iterrows():
    if row["Lifestyle_Category"] == "Healthy":
        sampled = healthy_meals.sample(1, replace=True).iloc[0]
    else:
        sampled = unhealthy_meals.sample(1, replace=True).iloc[0]
    
    assigned_meals.append(sampled.to_dict())

# Create merged dataset
assigned_meals_df = pd.DataFrame(assigned_meals)
merged_df = pd.concat([mental_df.reset_index(drop=True), assigned_meals_df.reset_index(drop=True)], axis=1)

merged_df.to_csv("person_level_merged.csv", index=False)


In [3]:
# Quick check
print(merged_df.shape)   # See how many rows and columns
print(merged_df.head())  # Preview first 5 rows


(290051, 43)
   Gender        Country Occupation self_employed family_history treatment  \
0  Female  United States  Corporate            No             No       Yes   
1  Female  United States  Corporate            No            Yes       Yes   
2  Female  United States  Corporate            No            Yes       Yes   
3  Female  United States  Corporate            No            Yes       Yes   
4  Female  United States  Corporate            No            Yes       Yes   

  Days_Indoors  Growing_Stress  Changes_Habits Mental_Health_History  ...  \
0    1-14 days             1.0             0.0                   Yes  ...   
1    1-14 days             1.0             0.0                   Yes  ...   
2    1-14 days             1.0             0.0                   Yes  ...   
3    1-14 days             1.0             0.0                   Yes  ...   
4    1-14 days             1.0             0.0                   Yes  ...   

    Carbs   Fats sugar_g Calories             Name of E

In [5]:
# Stratified sample: take 10% from each Lifestyle_Category
sample_df = merged_df.groupby('Lifestyle_Category', group_keys=False).apply(
    lambda x: x.sample(frac=0.1, random_state=42)
)

print(sample_df['Lifestyle_Category'].value_counts())


Lifestyle_Category
Unhealthy    17820
Healthy      11185
Name: count, dtype: int64


C:\Users\prasa\AppData\Local\Temp\ipykernel_14064\2393810044.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample_df = merged_df.groupby('Lifestyle_Category', group_keys=False).apply(


In [6]:
# Save the stratified sample to a CSV file
sample_df.to_csv("personal_level_sample.csv", index=False)


In [10]:
from sklearn.model_selection import train_test_split

healthy = merged_df[merged_df['Lifestyle_Category'] == 'Healthy']
unhealthy = merged_df[merged_df['Lifestyle_Category'] == 'Unhealthy']

healthy_sample = healthy.sample(n=5000, random_state=42)
unhealthy_sample = unhealthy.sample(n=5000, random_state=42)

reduced_df = pd.concat([healthy_sample, unhealthy_sample])

In [12]:
reduced_df.to_csv("Reduced_sample_New.csv", index=False)

In [13]:
print(reduced_df['Lifestyle_Category'].value_counts())

Lifestyle_Category
Healthy      5000
Unhealthy    5000
Name: count, dtype: int64


In [15]:
reduced_df.to_csv("Reduced_Sample_New2.csv", index=False, encoding='utf-8')


In [16]:
reduced_df.to_csv("Reduced_Sample_New3.tsv", sep="\t", index=False)
